# Just-In-Time Compilation with Numba

The exercises in this notebook were inspired by the [Numba tutorials](https://github.com/gforsyth/numba_tutorial_scipy2017) from the 2017 SciPy conference.

## The `jit` Decorator: Just-In-Time Compilation

In [1]:
def naive_sum(arr):
    """
    Sum all of the elements of a 2D array 'arr'
    """
    dim0 = len(arr)
    dim1 = len(arr[0])

    my_sum = 0
    for i in range(dim0):
        for j in range(dim1):
            my_sum += arr[i][j]
    
    return my_sum

In [2]:
import numpy as np

test_arr = np.random.random((300,400))

---
### Exercises

1. Time how long it takes to run `naive_sum` on the `test_arr` without any changes from Numba

In [3]:
%timeit naive_sum(test_arr)

29.5 ms ± 27.5 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


2. Redefine `naive_sum` but add Numba's `@jit` decorator just before the function definition. That is:

    ```python
    from numba import jit

    @jit
    def naive_sum(arr):
        ...
    ```

In [4]:
from numba import jit

@jit()
def naive_sum(arr):
    """
    Sum all of the elements of a 2D array 'arr'
    """
    dim0 = len(arr)
    dim1 = len(arr[0])

    my_sum = 0
    for i in range(dim0):
        for j in range(dim1):
            my_sum += arr[i][j]
    
    return my_sum

3. Numba adds some additional overhead the first time we run a `jit`-ified function because it must compile that function. Let's figure out how much additional time Numba's compilation actually adds. Use `time.perf_counter` to time how long **one execution** of our `jit`-ified `naive_sum` takes. This first execution will force Numba to compile `naive_sum`.

In [5]:
from time import perf_counter

start = perf_counter()
naive_sum(test_arr)
time = perf_counter() - start
print("Compilation took", time, "s")

Compilation took 3.9029715694487095 s


4. Now that Numba has compiled `naive_sum` once, **it will be available in memory until we restart the Python kernel**. Time how long the cached, pre-compiled `naive_sum` takes to run using either `perf_counter` or `timeit`. What factor of speedup does Numba provide?

In [6]:
%timeit naive_sum(test_arr)

143 μs ± 34.5 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [7]:
start = perf_counter()
naive_sum(test_arr)
print(perf_counter() - start)

0.0002624616026878357


A 100 times (or more) depending on timing method

5. We can accomplish the same task as `naive_sum` by using NumPy's own `sum` functions. Time how long it takes to sum `test_arr` using NumPy's `sum` function and compare this to the `jit`-ified version of `naive_sum`. Which is faster? Can Numba fully replace NumPy for performance?

In [8]:
%timeit test_arr.sum()

40.1 μs ± 50.7 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [9]:
%timeit np.sum(test_arr)

37.3 μs ± 47.2 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


The NumPy sum is still faster, so Numba cannot beat out NumPy's optimizations.

6. So far we've used the `naive_sum` function to operate on a NumPy array full of numbers, but the original code also supports summing a list of lists:

    ```python
    lst_arr = [[i*j for i in range(5)] for j in range(4)]
    naive_sum(lst_arr) # answer: 60
    ```

    Try running the `jit`-ified `naive_sum` on `lst_arr`. You should see several `NumbaWarning`s followed by an error. Recall that Numba has two modes -- `object` and `nopython` -- and use this to explain what Numba is trying to do with `naive_sum`.

In [10]:
lst_arr = [[i*j for i in range(5)] for j in range(4)]
naive_sum(lst_arr)

TypeError: cannot reflect element of reflected container: reflected list(reflected list(int64)<iv=None>)<iv=None>


Numba is falling back to object mode (before eventually still failing to compile)

7. If we could run `naive_sum` on a 2D NumPy array but *not* on a nested "2D" list, what does this tell you about Numba's limitations?

Numba works best with NumPy

---
## Parallelization with `@jit`

Numba's `jit` decorator has a `parallel` argument that, when set to `True`, tells Numba to try running your code on multiple processors/CPUs. Unlike most of the methods we have worked with so far, Numba will use *threads* for parallelization rather than processes. Numba's ability to compile portions of our code mean it can avoid Python's Global Interpreter Lock, which is what normally prevents use from using threads effectively.

The `parallel=True` option will try to parallelize your function automatically if it contains [supported operations](https://numba.readthedocs.io/en/stable/user/parallel.html#supported-operations). This parallelism can be a way to gain additional speedup on your function - even if `@jit` doesn't do much on its own.

Consider the following function and moderately large arrays for the following exercises:


In [11]:
def do_trig(x, y):
    z = np.sin(x**2) + np.cos(y)
    return z

x = np.random.random((1000, 1000))
y = np.random.random((1000, 1000))

---
### Exercises

1. Time how long it takes to run `do_trig(x,y)` without any adjustments.

In [12]:
%timeit do_trig(x,y)

28.8 ms ± 44.6 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


2. Now add the `@jit` decorator to `do_trig`. Don't add parallelism just yet! Time how long the function takes now; use `timeit` so that the compile time is hidden by multiple runs. There shouldn't be much change, because NumPy already relies on compiled libraries.

In [13]:
@jit
def do_trig(x, y):
    z = np.sin(x**2) + np.cos(y)
    return z

In [14]:
%timeit do_trig(x,y)

23.3 ms ± 1.56 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


3. Now, add `@jit(parallel=True)` to `do_trig` and time how long it takes to run with `timeit`. How much faster is the parallel version compared to the original?

In [15]:
@jit(parallel=True)
def do_trig(x, y):
    z = np.sin(x**2) + np.cos(y)
    return z

In [17]:
%timeit do_trig(x,y)

2.7 ms ± 68.3 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


About 10 times faster

---
### Parallelizing Over Loops

We can tell Numba to parallelize over iterations of a `for` loop using the `prange` iterator. This function is very similar to `range` except that it splits up the iterations of a `for` loop across multiple CPUs. This makes it very similar to how we used  `Pool.map()`, except that Numba will automatically handle any implied reductions (see the below example). If the `@jit` decorator is not applied, `prange` behaves just like `range`.

**Warning:** When using `prange`, it is up to you, the programmer, to make sure that each iteration is independent (except for implied reductions).

Time how long it takes to run the following function with and without the `@jit(parallel=True)` decorator.

In [ ]:
from numba import prange, jit

#@jit(parallel=True)
def prange_test(A):

    s = 0
    # Without "parallel=True" in the jit-decorator
    # the prange statement is equivalent to range
    for i in prange(A.shape[0]):
        s += A[i] # implied sum reduction

    return s

arr = np.arange(10000)
print(prange_test(arr))

49995000


In [ ]:
%timeit prange_test(arr)

1.08 ms ± 15.9 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


1 ms without jit and 10 us with

---
## The `vectorize` Decorator: Custom Element-by-Element NumPy Functions

One of the strengths of NumPy is its highly optimized functions that perform the same operation on every element of the array (e.g. `np.power`, `np.log10`). NumPy refers to these as "universal functions" or "ufuncs" for short. Such ufuncs can take advantage of the vectorization we learned about in Week 2. They are written in C to increase their speed and then linked into Python through a NumPy-specific interface.

Writing your own ufuncs with pure NumPy is a [convoluted process](https://numpy.org/doc/stable/user/c-info.ufunc-tutorial.html) and requires knowing C. Thankfully, Numba makes it incredibly easy with the `@vectorize` decorator.

Take the following function as an example:

In [18]:
import math

def sigmoid(x):
    return 1/(1+math.exp(-x))

This function currently only operates on a single (scalar) value, as the built-in `math` module doesn't understand arrays or lists (you can try it yourself and see the error that results). In the following exercises you'll explore using `sigmoid` with and without the `@vectorize` decorator.

---
### Exercises

1. Time how long it takes to create a new array or list by applying the `sigmoid` function to every integer in `range(-200,200)`

    **Hint:** Recall what you learned in Week 1 and pick what you think is the most appropriate approach

In [19]:
%timeit new_arr = [sigmoid(i) for i in range(-200, 200)]

64.4 μs ± 196 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


2. Redefine the `sigmoid` function and add Numba's `@vectorize` decorator. You'll first need to import `vectorize` similar to how we imported `jit`.

In [20]:
from numba import vectorize

@vectorize
def sigmoid(x):
    return 1/(1+math.exp(-x))

3. Time how long it takes to run your new vectorized `sigmoid` function on the array below. By what factor is it faster?

In [21]:
arr = np.arange(-200, 200, dtype=int)

In [22]:
%timeit sigmoid(arr)

3.43 μs ± 20.7 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


22 times faster with timeit